# URJA Yield — Grid Price Forecast (LSTM)
**Colab T4 → Latitude inference.** Replaces `sin()` mock in `refresh_pricing.py` with a 24h forecast.
**Run all → download `price_forecast.onnx` → Latitude serves it.**


In [ ]:
!nvidia-smi | head -3
!test -d URJA || git clone https://github.com/ravikumarve/URJA.git --depth 1
%cd URJA
!pip -q install torch --index-url https://download.pytorch.org/whl/cu121 --progress-bar off 2>&1 | tail -1
!pip -q install pandas scikit-learn matplotlib --progress-bar off 2>&1 | tail -1
print("✅ ready")

In [ ]:
import pandas as pd, numpy as np, torch
np.random.seed(42); torch.manual_seed(42)
print("CUDA", torch.cuda.is_available())
hours = 24*365*2  # 2 years hourly
t = np.arange(hours)
price = 3.5 * (0.5 + 2.0*np.maximum(0, np.sin((t%24 -6)*3.14159/13))) + np.random.normal(0,0.3,hours)
price = np.clip(price, 0.8, 8.5)
df = pd.DataFrame({"hour": t%24, "dow": (t//24)%7, "price": price})
df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import torch.nn as nn

SEQ=24
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[["price","hour","dow"]])
X,y=[],[]
for i in range(len(scaled)-SEQ-24):
    X.append(scaled[i:i+SEQ])
    y.append(scaled[i+SEQ:i+SEQ+24,0])
X,y = np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)
print(X.shape, y.shape)

class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(3,32,batch_first=True)
        self.fc=nn.Linear(32,24)
    def forward(self,x):
        _,(h,_)=self.lstm(x)
        return self.fc(h[-1])

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=LSTM().to(device)
opt=torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn=nn.MSELoss()
loader=torch.utils.data.DataLoader(list(zip(X,y)), batch_size=256, shuffle=True)
for epoch in range(8):
    tot=0
    for xb,yb in loader:
        xb,yb=xb.to(device), yb.to(device)
        opt.zero_grad(); loss=loss_fn(model(xb), yb); loss.backward(); opt.step()
        tot+=loss.item()
    print(f"epoch {epoch+1} loss {tot/len(loader):.5f}")

In [ ]:
import matplotlib.pyplot as plt
model.eval()
with torch.no_grad():
    pred = model(torch.tensor(X[-1:]).to(device)).cpu().numpy()[0]
    true = y[-1]
pred_inv = scaler.inverse_transform(np.column_stack([pred, np.zeros((24,2))]))[:,0]
true_inv = scaler.inverse_transform(np.column_stack([true, np.zeros((24,2))]))[:,0]
plt.plot(true_inv, label="true", color="#ffb703")
plt.plot(pred_inv, label="pred", color="#ff5e00")
plt.legend(); plt.title("24h forecast (₹/kWh)"); plt.show()
print(f"MAE ₹{abs(pred_inv-true_inv).mean():.2f}/kWh")

In [ ]:
torch.save(model.state_dict(), "price_forecast.pt")
import pickle
Path = __import__("pathlib").Path
Path("scaler.pkl").write_bytes(pickle.dumps(scaler))
print("price_forecast.pt", Path("price_forecast.pt").stat().st_size//1024, "KB")
from google.colab import files; files.download("price_forecast.pt"); files.download("scaler.pkl")
print("↓ move to backend/app/services/price_forecast.pt")